In [ ]:
# import necessary packages
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import AllChem
import numpy as np
from sklearn import linear_model
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# define colors used in this work
paton_hot_pink = '#d00d44'
rosewood_plum = '#9B6B8E'     
rosewood_sage = '#7BA882'     
rosewood_gold = '#D4A574'       
rosewood_slate = '#5B7FA6'     
rosewood_rose = '#C9858C'        
rosewood_charcoal = '#4A4A4A'    
rosewood_cream = '#F5EDE0' 
tol_grey = '#BBBBBB'

# define plotting parameters
sns.set(context='talk', style='ticks',
        color_codes=True, rc={'legend.frameon': False})
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
plt.rcParams["font.family"] = 'Avenir'
plt.rcParams.update({'font.size': 24})
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# read in the full dataset
full_dataset = pd.read_csv("hammett_predictions.csv")
full_dataset = full_dataset.drop(columns=['nitrile_smiles'])

# remove H from the para substituents so it is not double-counted
full_dataset = full_dataset[full_dataset['nitrile_id'] != 'para_nitrile_032']

# identify substituents which performed well for the full model
x = full_dataset['hammett_parameter'].values
y = full_dataset['predicted_hammett'].values

regr = linear_model.LinearRegression()

# Fit the model
regr.fit(x.reshape(-1, 1), y)

# Calculate residuals
residuals = y - regr.predict(x.reshape(-1, 1)).flatten()

# Define a threshold for outliers (e.g., 1 standard deviations from the mean)
threshold = 0.5 * np.std(residuals)

# Identify outliers
outliers = np.abs(residuals) > threshold

df_best = full_dataset[~outliers]

In [ ]:
# identify those substituents for which both meta and para performed well
df_best['nitrile_number'] = df_best['nitrile_id'].str.split('_').str[2]
df_best_filtered = df_best[df_best['nitrile_number'].map(df_best['nitrile_number'].value_counts()) >= 2]

In [ ]:
# Add number of rotatable bonds to the dataframe
smis = df_best_filtered['Substituent']
mols = [Chem.MolFromSmiles(smi) for smi in smis]
rot_bonds = [rdMolDescriptors.CalcNumRotatableBonds(mol) if mol else np.nan for mol in mols]
df_best_filtered['rot_bonds'] = rot_bonds

# filter out substituents with > 2 rotatable bonds in order to reduce number of substituents and computational cost
df_best_filtered = df_best_filtered[df_best_filtered['rot_bonds'] < 2]

In [ ]:
# determine the performance of the full model for just the best-performing substituents
x = df_best_filtered['hammett_parameter'].values.reshape(-1, 1)
y = df_best_filtered['predicted_hammett'].values.reshape(-1, 1)

regrs = linear_model.LinearRegression()
regrs.fit(x, y)
r_value = r2_score(y, regrs.predict(x))

mae = mean_absolute_error(x, y)

print(f"R^2 for best-performing substituents: {r_value:.2f}")
print(f"MAE for best-performing substituents: {mae:.2f}")

In [ ]:
# remove unncessary columns
df_best_filtered = df_best_filtered.drop(columns=['cm5_charge', 'predicted_hammett', 'rot_bonds','F','R'])

In [ ]:
# separate the meta and para substituents
best_meta = df_best_filtered['nitrile_id'].str.contains('meta', na=False)
best_meta = df_best_filtered[best_meta]
best_meta.rename(columns={'nitrile_id': 'meta_nitrile_id'}, inplace=True)
best_meta.rename(columns={'hammett_parameter': 'sigma_m'}, inplace=True)

best_para = df_best_filtered['nitrile_id'].str.contains('para', na=False)
best_para = df_best_filtered[best_para]
best_para.rename(columns={'nitrile_id': 'para_nitrile_id'}, inplace=True)
best_para.rename(columns={'hammett_parameter': 'sigma_p'}, inplace=True)

In [ ]:
# combine the meta and para into one dataframe
df_best_concat = pd.merge(best_meta, best_para, how='left', on='nitrile_number')
df_best_concat = df_best_concat.drop(columns=['Substituent_x'])
df_best_concat = df_best_concat.rename(columns={'Substituent_y': 'Substituent'})
df_best_concat.head()

In [ ]:
# OPTIONAL – save the best substituents to a csv file
df_best_concat.to_csv("best_substituents.csv", index=False)

In [ ]:
# make mol objects for the substituents
smis = df_best_concat['Substituent']
mol_objects = [Chem.MolFromSmiles(smi) for smi in smis]
df_best_concat['mols'] = mol_objects

In [ ]:
# define benzonitrile and the reaction to add the meta substituent to benzonitrile
benzonitrile = Chem.MolFromSmiles('N#CC1=CC=CC=C1')
meta_rxn = AllChem.ReactionFromSmarts('[N:1]#[C:2]-[c,C:3]1[c,C:4][c,C:5][c,C:6][c,C:7][c,C:8]1.[At]-[*:9]>>[N:1]#[C:2][c3]1[c8][c7][c6][c5]([c4]1)[*:9]')
meta_rxn

In [ ]:
# create the meta-substituted products
meta_products = []
for subst in df_best_concat['mols']:
    print(f"Reacting benzonitrile with {subst}")
    result = meta_rxn.RunReactants((benzonitrile, subst))
    print(f"Result: {result}")
    if result and result[0]:  # Check if there is at least one product
        meta_products.append(result[0][0])
    else:
        meta_products.append(None)  # Or handle the case as needed

meta_smis = [Chem.MolToSmiles(mol) if mol is not None else '' for mol in meta_products]
df_best_concat['single_meta_substituents'] = meta_smis
df_best_concat = df_best_concat.drop(columns=['mols'])

In [ ]:
# save the meta-substituted nitriles, we will need this as a csv
df_best_concat.to_csv('best_with_meta_substituted_nitriles.csv', index=False)

In [ ]:
# read in the meta-substituted nitriles to make the 3,4-disubstituted benzonitriles
meta = 'best_with_meta_substituted_nitriles.csv'

df_meta_para = pd.read_csv(meta)
df_meta_para.rename(columns={'nitrile_number': 'substituent_id'}, inplace=True)

In [ ]:
# make mol objects for the meta-substituted nitriles
smis = df_meta_para['single_meta_substituents']
mol_objects = [Chem.MolFromSmiles(smi) for smi in smis]
df_meta_para['meta_mols'] = mol_objects

In [ ]:
# define the SMARTS reaction to make the 3,4 disubstituted benzonitriles
rxn_3_4 = AllChem.ReactionFromSmarts('[N:1]#[C:2][c3]1[c8][c7][c6][c5]([c4]1)[*:9].[At]-[*:10]>>[N:1]#[C:2]-[c:3]1[c:4][c:5][c:6]([c:7]([c:8]1)-[*:9])-[*:10]')
rxn_3_4

In [ ]:
# make the 3,4 disubstituted benzonitriles 
for subst_id, subst in zip(df_meta_para['substituent_id'], df_meta_para['Substituent']):
    subst_mol = Chem.MolFromSmiles(subst)
    if subst_mol:
        column_name = f"para_substituent_{subst_id}"
        para_products = []
        for meta_mol in df_meta_para['meta_mols']:
            if meta_mol:
                result = rxn_3_4.RunReactants((meta_mol, subst_mol))
                if result and result[0]:  # Check if there is at least one product
                    para_products.append(result[0][0])
                else:
                    para_products.append(None)
            else:
                para_products.append(None)
        df_meta_para[column_name] = para_products

In [ ]:
df_meta_para = df_meta_para.drop(columns=['meta_mols','single_meta_substituents'])

# convert the 3,4-disubstituted products to SMILES strings
for col in df_meta_para.columns:
    if col.startswith('para_substituent_'):
        df_meta_para[col] = df_meta_para[col].apply(lambda x: Chem.MolToSmiles(x) if x is not None else '')

In [ ]:
# create IDS for which the first substituent listed is the meta substituent and the second is the para substituent 
smiles_data = []
for index, row in df_meta_para.iterrows():
    substituent_id = row['substituent_id']
    for col in df_meta_para.columns:
        if col.startswith('para_substituent_'):
            smiles = row[col]
            name = f"{substituent_id}_{col.split('_')[-1]}"
            smiles_data.append({'Name': name, 'SMILES': smiles})

new_df_meta_para = pd.DataFrame(smiles_data)

In [ ]:
# add the appropriate sigma_m values
new_df_meta_para['substituent_id'] = new_df_meta_para['Name'].apply(lambda name: int(name.split('_')[0]))
new_df_meta_para['substituent_id'] = new_df_meta_para['substituent_id'].apply(lambda x: f"{int(x):03}")

df_values = df_meta_para[['substituent_id', 'sigma_m']]

new_df_meta_para = new_df_meta_para.merge(df_values, on='substituent_id', how='left')

new_df_meta_para = new_df_meta_para.drop(columns=['substituent_id'])

In [ ]:
# add the appropriate sigma_p values
new_df_meta_para['substituent_id'] = new_df_meta_para['Name'].apply(lambda name: int(name.split('_')[1]))
new_df_meta_para['substituent_id'] = new_df_meta_para['substituent_id'].apply(lambda x: f"{int(x):03}")

df_values = df_meta_para[['substituent_id', 'sigma_p']]

new_df_meta_para = new_df_meta_para.merge(df_values, on='substituent_id', how='left')

new_df_meta_para = new_df_meta_para.drop(columns=['substituent_id'])

In [ ]:
# add the sum of the hammett parameters to the dataframe
new_df_meta_para['hammett_sum'] = new_df_meta_para['sigma_m'] + new_df_meta_para['sigma_p']

# save the 3,4-disubstituted nitriles to a csv file
new_df_meta_para.to_csv('3_4_substituted_nitriles.csv', index=False)

We will now repeat all the above steps but for the 3,5 disubstituted benzonitriles.

In [ ]:
meta = 'best_with_meta_substituted_nitriles.csv'

df_meta_meta = pd.read_csv(meta)

In [ ]:
smis = df_meta_meta['single_meta_substituents']
mol_objects = [Chem.MolFromSmiles(smi) for smi in smis]
df_meta_meta['meta_mols'] = mol_objects

df_meta_meta.rename(columns={'nitrile_number': 'substituent_id'}, inplace=True)

In [ ]:
rxn_3_5 = AllChem.ReactionFromSmarts('[N:1]#[C:2][c3]1[c8][c7][c6][c5]([c4]1)[*:9].[At]-[*:10]>>[N:1]#[C:2][c3]1[c8][c7]([c6][c5]([c4]1)[*:10])[*:9]')
rxn_3_5

In [ ]:
for subst_id, subst in zip(df_meta_meta['substituent_id'], df_meta_meta['Substituent']):
    subst_mol = Chem.MolFromSmiles(subst)
    if subst_mol:
        column_name = f"meta_substituent_{subst_id}"
        meta_products = []
        for meta_mol in df_meta_meta['meta_mols']:
            if meta_mol:
                result = rxn_3_5.RunReactants((meta_mol, subst_mol))
                if result and result[0]:  # Check if there is at least one product
                    meta_products.append(result[0][0])
                else:
                    meta_products.append(None)
            else:
                meta_products.append(None)
        df_meta_meta[column_name] = meta_products

In [ ]:
df_meta_meta = df_meta_meta.drop(columns=['meta_mols'])

for col in df_meta_meta.columns:
    if col.startswith('meta_substituent_'):
        df_meta_meta[col] = df_meta_meta[col].apply(lambda x: Chem.MolToSmiles(x) if x is not None else '')

In [ ]:
smiles_data = []
for index, row in df_meta_meta.iterrows():
    substituent_id = row['substituent_id']
    for col in df_meta_meta.columns:
        if col.startswith('meta_substituent_'):
            smiles = row[col]
            name = f"{substituent_id}_{col.split('_')[-1]}"
            smiles_data.append({'Name': name, 'SMILES': smiles})

new_df_meta_meta = pd.DataFrame(smiles_data)

In [ ]:
new_df_meta_meta['substituent_id'] = new_df_meta_meta['Name'].apply(lambda name: int(name.split('_')[0]))
new_df_meta_meta['substituent_id'] = new_df_meta_meta['substituent_id'].apply(lambda x: f"{int(x):03}")

df_values = df_meta_meta[['substituent_id', 'sigma_m']]

new_df_meta_meta = new_df_meta_meta.merge(df_values, on='substituent_id', how='left')

new_df_meta_meta = new_df_meta_meta.drop(columns=['substituent_id'])

new_df_meta_meta['sigma_m_1'] = new_df_meta_meta['sigma_m']

new_df_meta_meta.drop(columns=['sigma_m'], inplace=True)

In [ ]:
new_df_meta_meta['substituent_id'] = new_df_meta_meta['Name'].apply(lambda name: int(name.split('_')[1]))
new_df_meta_meta['substituent_id'] = new_df_meta_meta['substituent_id'].apply(lambda x: f"{int(x):03}")

df_values = df_meta_meta[['substituent_id', 'sigma_m']]

new_df_meta_meta = new_df_meta_meta.merge(df_values, on='substituent_id', how='left')

new_df_meta_meta = new_df_meta_meta.drop(columns=['substituent_id'])

new_df_meta_meta['sigma_m_2'] = new_df_meta_meta['sigma_m']

new_df_meta_meta.drop(columns=['sigma_m'], inplace=True)

In [ ]:
new_df_meta_meta['hammett_sum'] = new_df_meta_meta['sigma_m_1'] + new_df_meta_meta['sigma_m_2']

new_df_meta_meta.to_csv('3_5_substituted_nitriles.csv', index=False)

In [ ]:
# select the substituents for testing
meta_para = '3_4_substituted_nitriles.csv'
meta_meta = '3_5_substituted_nitriles.csv'

df_3_4 = pd.read_csv(meta_para)
df_3_5 = pd.read_csv(meta_meta)

In [ ]:
# randomly select 100 disubstituted benzonitriles
selected_3_4 = df_3_4.sample(n=100, random_state=12)

selected_3_4.to_csv('selected_3_4_substituted_nitriles.csv', index=False)

In [ ]:
# select the same substituent patterns for the 3,5-disubstituted benzonitriles
selected_3_5 = df_3_5[df_3_5['Name'].isin(selected_3_4['Name'])]

selected_3_5.to_csv('selected_3_5_substituted_nitriles.csv', index=False)

In [ ]:
# prepare csvs for AQME conformer search and QM input generation for the 3,4-disubstituted benzonitriles
selected_3_4 = 'selected_3_4_substituted_nitriles.csv'
aqme_3_4 = pd.read_csv(selected_3_4)

aqme_3_4 = aqme_3_4.drop(columns=['sigma_m', 'sigma_p', 'hammett_sum'])
aqme_3_4.rename(columns={'Name': 'code_name'}, inplace=True)

# create descriptive file names
aqme_3_4['code_name'] = aqme_3_4['code_name'].apply(lambda name: f"meta{name.split('_')[0]}_para{name.split('_')[1]}_nitrile")

aqme_3_4.to_csv('aqme_3_4.csv', index=False)

In [ ]:
# prepare csvs for AQME conformer search and QM input generation for the 3,5-disubstituted benzonitriles
selected_3_5 = 'selected_3_5_substituted_nitriles.csv'
aqme_3_5 = pd.read_csv(selected_3_5)

aqme_3_5 = aqme_3_5.drop(columns=['sigma_m_1', 'sigma_m_2', 'hammett_sum'])
aqme_3_5.rename(columns={'Name': 'code_name'}, inplace=True)

aqme_3_5['code_name'] = aqme_3_5['code_name'].apply(lambda name: f"meta{name.split('_')[0]}_meta{name.split('_')[1]}_nitrile")

aqme_3_5.to_csv('aqme_3_5.csv', index=False)

To the randomly selected disubstituted benzonitriles, manually selected substituents with very negative Hammett values were added. This is to ensure a reasonable spread of data for the final model. These SMILES and names were added to the appropriate csvs.

Additionally, polysubstituted benzonitriles with experimentally determined Hammett parameters were also added to the dataset. These have been given names in the format 'nitrile_no_##', with numbers that correspond to the experimental paper referenced in the associated paper.

In [ ]:
# generate conformers and optimization inputs using AQME 
! python -m aqme --csearch --program rdkit --input aqme_3_4.csv 
! mv CSEARCH conformers_3_4
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2SVP emp=GD3BJ opt freq" --files "conformers_3_4/*.sdf"
! mv QCALC optimization_inputs_3_4

In [ ]:
# generate conformers and optimization inputs using AQME
! python -m aqme --csearch --program rdkit --input aqme_3_5.csv 
! mv CSEARCH conformers_3_5
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2SVP emp=GD3BJ opt freq" --files "conformers_3_5/*.sdf"
! mv QCALC optimization_inputs_3_5

Run the Gaussian jobs in the appropriate location, then return the output files to appropriately named directories.

In [ ]:
# check Gaussian output files for errors using AQME
! python -m aqme --qcorr --program gaussian --files "optimization_outputs_3_4/*.log"

In [ ]:
# check Gaussian output files for errors using AQME
! python -m aqme --qcorr --program gaussian --files "optimization_outputs_3_5/*.log"

In [ ]:
# prepare single point files to obtain charges
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2QZVPP emp=GD3BJ pop=(Hirshfeld)" --files "optimization_outputs_3_4/success/*.log"
! mv QCALC single_point_inputs_3_4

In [ ]:
# prepare single point files to obtain charges
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2QZVPP emp=GD3BJ pop=(Hirshfeld)" --files "optimization_outputs_3_5/success/*.log"
! mv QCALC single_point_inputs_3_5

In [ ]:
# check Gaussian output files for errors using AQME
! python -m aqme --qcorr --program gaussian --files "single_point_outputs_3_4/*.log"

In [ ]:
# check Gaussian output files for errors using AQME
! python -m aqme --qcorr --program gaussian --files "single_point_outputs_3_5/*.log"

In [ ]:
# OPTIONAL – make new directories and move files appropriately 
! mkdir spc_3_4
! cp single_point_outputs_3_4/success/SP_calcs/*.log spc_3_4/
! mkdir opt_3_4
! cp optimization_outputs_3_4/success/*.log opt_3_4/

! mkdir spc_3_5
! cp single_point_outputs_3_5/success/SP_calcs/*.log spc_3_5/
! mkdir opt_3_5
! cp optimization_outputs_3_5/success/*.log opt_3_5/

In [ ]:
# CHANGE ENVIRONMENT TO APPROPRIATE ENVIRONMENT FOR MOLDSCRIPT  
# use molDscript to parse atomic charges 
! python -m moldscript --opt opt_3_4 --spc spc_3_4 --charges spc_3_4 --substructure 'cC#N' --boltz --min_max --lowe
! mkdir charges_3_4
! mv *_level.csv charges_3_4
! mv MOLDSCRIPT*.dat charges_3_4

In [ ]:
# use molDscript to parse atomic charges 
! python -m moldscript --opt opt_3_5 --spc spc_3_5 --charges spc_3_5 --substructure 'cC#N' --boltz --min_max --lowe
! mkdir charges_3_5
! mv *_level.csv charges_3_5
! mv MOLDSCRIPT*.dat charges_3_5

In [ ]:
# clean molDscript data to obtain only the desired charges in the same method as the monosubstituted benzonitriles
charges_3_4 = 'charges_3_4/lowest_energy_atom_level.csv'
df_3_4 = pd.read_csv(charges_3_4)

df_3_4_N = df_3_4[df_3_4['atom_type'] == 'N']
df_3_4_N = df_3_4_N.sort_values(by='filename')
df_3_4_N = df_3_4_N.rename(columns={'filename': 'Name'})
df_3_4_N['Name'] = df_3_4_N['Name'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[1], x.split('_')[2].zfill(3)]))

In [ ]:
# add in the experimental Hammett sum information
selected_3_4 = 'selected_3_4_substituted_nitriles.csv'
exp_values_3_4 = pd.read_csv(selected_3_4)

exp_values_3_4 = exp_values_3_4[['Name', 'sigma_m', 'sigma_p', 'hammett_sum', 'SMILES']]
exp_values_3_4['Name'] = exp_values_3_4['Name'].apply(lambda name: f"meta{name.split('_')[0]}_para{name.split('_')[1]}_nitrile")
compare_3_4 = pd.merge(df_3_4_N, exp_values_3_4, on='Name')

In [ ]:
# identify any molecules for which more than one charge for N was obtained
repeats_3_4 = compare_3_4[compare_3_4.duplicated('Name', keep=False)]
repeats_3_4_df = repeats_3_4.copy()
repeats_3_4_df

In [ ]:
# remove incorrectly parsed N atoms from the dataset after manual examination
compare_3_4 = compare_3_4[~((compare_3_4['Name'] == 'meta003_para125_nitrile') & (compare_3_4['atom_index'] != 1))]
compare_3_4 = compare_3_4[~((compare_3_4['Name'] == 'meta011_para031_nitrile') & (compare_3_4['atom_index'] != 1))]

compare_3_4.to_csv('charges_3_4.csv', index=False)

We will now repeat the data processing for the 3,5-disubstituted benzonitriles.

In [ ]:
# clean molDscript data to obtain only the desired charges
charges_3_5 = 'charges_3_5/lowest_energy_atom_level.csv'
df_3_5 = pd.read_csv(charges_3_5)
df_3_5_N = df_3_5[df_3_5['atom_type'] == 'N']
df_3_5_N = df_3_5_N.sort_values(by='filename')
df_3_5_N = df_3_5_N.rename(columns={'filename': 'Name'})

In [ ]:
selected_3_5 = 'selected_3_5_substituted_nitriles.csv'
exp_values_3_5 = pd.read_csv(selected_3_5)
exp_values_3_5['Name'] = exp_values_3_5['Name'].apply(lambda name: f"meta{name.split('_')[0]}_meta{name.split('_')[1]}_nitrile")

In [ ]:
exp_values_3_5 = exp_values_3_5[['Name', 'sigma_m_1', 'sigma_m_2', 'hammett_sum', 'SMILES']]
compare_3_5 = pd.merge(df_3_5_N, exp_values_3_5, on='Name')

In [ ]:
repeats_3_5 = compare_3_5[compare_3_5.duplicated('Name', keep=False)]
repeats_3_5_df = repeats_3_5.copy()
repeats_3_5_df

In [ ]:
compare_3_5.to_csv('charges_3_5.csv', index=False) 

In [ ]:
# predict hammett constants for the 3,4- and 3,5-disubstituted benzonitriles using the equation obtained from the monosubstituted benzonitriles
compare_3_4['predicted_hammett'] = 34.53 * compare_3_4['cm5_charge'] + 11.90

compare_3_5['predicted_hammett'] = 34.53 * compare_3_5['cm5_charge'] + 11.90